# Add LSST Photometric Calibration Parameters to Light Curves

This notebook enriches the light-curve files produced by
`02_MergeLCsourceswithMJD.ipynb` with per-exposure photometric calibration
information retrieved from the Butler.

## Important note on the detector column

The `detector` column in the input light-curve files is **unreliable** (filled
with -1). The correct detector is resolved automatically by the Butler through
a spatial query on the sky position of each star.

## Strategy

The LSSTCam focal plane has 189 detectors. During a single visit, different
stars fall on different detectors. Therefore the loop runs **star by star**:

1. For each star (identified by its fixed sky position `src_ra`, `src_dec`):
   - Issue **one single `butler.query_datasets()` call per band** with a wide
     timespan and a spatial constraint `OVERLAPS POINT(ra, dec)`. This returns
     references for **all visits** in which that star was observed in that band,
     each pointing to the correct detector automatically.
   - Build a lookup dict `visit → dataset_ref` from these references.
   - Iterate over the rows of the star's light curve. For each row, find the
     matching ref by visit number, then load the PVI and extract PhotoCalib +
     detector info — **unless the result is already in the cache**.

2. A global cache keyed on `(band, visit, detector_id)` avoids reloading the
   same PVI when two stars happen to share a visit on the same detector.
   The `detector_id` in the key is obtained from the dataset ref's dataId
   (not from the unreliable LC column).

## What is added per data point

| New column | Source | Meaning |
|---|---|---|
| `calib_mean` | `photocal.getCalibrationMean()` | Mean calibration factor over the CCD [nJy / ADU] |
| `calib_err` | `photocal.getCalibrationErr()` | Uncertainty on the mean calibration factor |
| `calib_local` | `photocal.getLocalCalibration(Point2D(x, y))` | Local calibration factor at the source pixel position |
| `zeropoint` | `2.5 * log10(photocal.getInstFluxAtZeroMagnitude())` | AB zero-point magnitude |
| `visit_from_image` | `pvi.visitInfo.id` | Visit id embedded in the image (sanity check vs `visit`) |
| `detector_id` | `pvi.getDetector().getId()` | Detector number (replaces the -1 placeholder) |
| `detector_name` | `pvi.getDetector().getName()` | Detector name (e.g. `R22_S11`) |

## Input
- `data_MergeVisits_02_out/all_stars_lightcurves_mjd.csv` (global)
- `data_MergeVisits_02_out/per_star/*_lc_mjd.csv` (per-star)

## Output
- `data_AddCalib_05_out/all_stars_lightcurves_calib.csv` + `.parquet`
- `data_AddCalib_05_out/per_star/*_lc_calib.csv` + `.parquet`

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-29
- **Last update:** 2026-06-29


## 1. Imports

In [ ]:
import gc
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from astropy.time import Time

from lsst.daf.butler import Butler, Timespan
from lsst.geom import Point2D

from libExtractLightcurves import safe_name, find_col, dataset_type_exists

## 2. Logging setup

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Logging initialised.")

## 3. Configuration

In [ ]:
# ── Notebook tag ─────────────────────────────────────────────────────────────
NB_TAG = "AddCalib_05"

# ── Input: light curves with MJD (output of notebook 02) ─────────────────────
DIR_LC_IN = "./data_MergeVisits_02_out"
DIR_LC_PER_STAR_IN = os.path.join(DIR_LC_IN, "per_star")
GLOBAL_LC_FILE = "all_stars_lightcurves_mjd.csv"

# ── Output ────────────────────────────────────────────────────────────────────
DIR_DATA = f"./data_{NB_TAG}_out"
DIR_DATA_PER_STAR = os.path.join(DIR_DATA, "per_star")
DIR_FIGS = f"./figs_{NB_TAG}"

for d in [DIR_DATA, DIR_DATA_PER_STAR, DIR_FIGS]:
    os.makedirs(d, exist_ok=True)
    log.info("Directory ready: %s", d)

# ── Butler configuration ──────────────────────────────────────────────────────
repo = "dp2_prep"
collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

# ── Timespan: wide window covering all DP2 (used in spatial queries) ──────────
DATE_START = "2025-04-01T00:00:00"
DATE_STOP = "2026-07-01T00:00:00"
MJD_START = Time(DATE_START, format="isot", scale="utc").mjd
MJD_STOP = Time(DATE_STOP, format="isot", scale="utc").mjd
log.info("MJD window: [%.2f, %.2f]", MJD_START, MJD_STOP)

# ── Candidate dataset type names for the single-visit processed image ─────────
PVI_TABLE_CANDIDATES = [
    "preliminary_visit_image",
    "legacy_visit_image",
    "visit_image",
]

log.info("Configuration done.")

## 4. Initialise the Butler and select the PVI dataset type

In [ ]:
butler = Butler(repo, collections=collection)
registry = butler.registry
log.info("Butler initialised | repo: %s", repo)

In [ ]:
# List visit-image-related dataset types for information
all_pvi_types = [
    d.name for d in registry.queryDatasetTypes() if "visit" in d.name.lower() and "image" in d.name.lower()
]
print("visit-image-related dataset types in registry:")
for t in sorted(all_pvi_types):
    print(f"  {t}")

In [ ]:
# Select the first registered PVI dataset type
PVI_DATASET = None
for name in PVI_TABLE_CANDIDATES:
    if dataset_type_exists(butler, name):
        PVI_DATASET = name
        log.info("Selected PVI dataset type: '%s'", PVI_DATASET)
        break

if PVI_DATASET is None:
    raise RuntimeError(
        "No recognised PVI dataset type found in this Butler collection. "
        f"Candidates tried: {PVI_TABLE_CANDIDATES}"
    )

## 5. Load the global merged light-curve table

In [ ]:
lc_path = os.path.join(DIR_LC_IN, GLOBAL_LC_FILE)
log.info("Loading: %s", lc_path)
df_all = pd.read_csv(lc_path)
log.info("Shape: %s  |  columns: %s", df_all.shape, df_all.columns.tolist())
df_all.head(3)

In [ ]:
# Verify required columns.
# 'detector' is NOT required — unreliable (-1), replaced by Butler output.
REQUIRED_COLS = ["simbad_id", "visit", "src_ra", "src_dec", "x", "y", "band"]
missing = [c for c in REQUIRED_COLS if c not in df_all.columns]
if missing:
    raise ValueError(f"Required columns missing from input LC file: {missing}")
log.info("All required columns present.")

df_all["visit"] = df_all["visit"].astype(np.int64)

# List of unique stars
star_ids = df_all["simbad_id"].unique()
log.info("Number of unique stars: %d", len(star_ids))

## 6. Build the Timespan for Butler spatial queries

In [ ]:
t1 = Time(MJD_START, format="mjd", scale="tai")
t2 = Time(MJD_STOP, format="mjd", scale="tai")
timespan = Timespan(t1, t2)
log.info("Timespan for Butler queries: MJD [%.1f, %.1f]  (delta=%.0f days)", t1.mjd, t2.mjd, t2.mjd - t1.mjd)

## 7. Core processing: star-by-star loop

### Design

For each star and each band:
- **One `query_datasets` call** with the star's fixed sky position and a wide
  timespan returns references for *all* visits in which that star was observed
  in that band, each automatically pointing to the correct detector.
- Build a `visit_id → ref` lookup from these references.
- For each row in the star's light curve, look up the ref by visit number,
  then load the PVI and extract calib info — **skipped if already cached**.

### Global cache

```
calib_cache[(band, visit, detector_id)] = calib_dict
```

The `detector_id` key is obtained from `ref.dataId["detector"]` (reliable,
from the registry), not from the LC file.  If a second star happens to share
the same `(band, visit, detector)` the PVI is not reloaded.

In [ ]:
def load_pvi_calib(
    butler: Butler,
    ref,
) -> dict:
    """Load one PVI and extract PhotoCalib + detector scalars.

    Returns a dict with keys:
        calib_mean, calib_err, zeropoint,
        visit_from_image, detector_id, detector_name,
        photocal  (kept for per-row getLocalCalibration),
        calib_ok  (bool).
    """
    result = dict(
        calib_mean=np.nan,
        calib_err=np.nan,
        zeropoint=np.nan,
        visit_from_image=-1,
        detector_id=-1,
        detector_name="",
        photocal=None,
        calib_ok=False,
    )
    try:
        pvi = butler.get(ref)
        photocal = pvi.getPhotoCalib()
        det_obj = pvi.getDetector()

        result["calib_mean"] = photocal.getCalibrationMean()
        result["calib_err"] = photocal.getCalibrationErr()
        result["zeropoint"] = 2.5 * np.log10(photocal.getInstFluxAtZeroMagnitude())
        result["visit_from_image"] = int(pvi.visitInfo.id)
        result["detector_id"] = int(det_obj.getId())
        result["detector_name"] = str(det_obj.getName())
        result["photocal"] = photocal
        result["calib_ok"] = True
    except Exception as exc:
        log.warning("    load_pvi_calib failed for ref %s: %s", ref.dataId, exc)

    return result

In [ ]:
# Pre-allocate new columns in the global DataFrame
df_all["calib_mean"] = np.nan
df_all["calib_err"] = np.nan
df_all["calib_local"] = np.nan
df_all["zeropoint"] = np.nan
df_all["visit_from_image"] = -1
df_all["detector_id"] = -1
df_all["detector_name"] = ""

# Global cache: (band, visit, detector_id) → calib_dict
calib_cache: dict = {}

bands_in_data = sorted(df_all["band"].unique())
log.info("Bands present in data: %s", bands_in_data)

n_stars_ok = 0
n_stars_err = 0

for i_star, star_id in enumerate(star_ids):
    # Rows belonging to this star
    mask_star = df_all["simbad_id"] == star_id
    df_star = df_all.loc[mask_star]

    # Sky position of this star (fixed — use first row)
    ra_star = float(df_star["src_ra"].iloc[0])
    dec_star = float(df_star["src_dec"].iloc[0])

    log.info(
        "[%d/%d] Star: %-30s  (ra=%.5f, dec=%.5f)  %d rows",
        i_star + 1,
        len(star_ids),
        star_id,
        ra_star,
        dec_star,
        mask_star.sum(),
    )

    for band in bands_in_data:
        mask_band = mask_star & (df_all["band"] == band)
        if not mask_band.any():
            continue

        visits_for_star_band = df_all.loc[mask_band, "visit"].unique()
        log.info("  band=%-3s  %d visits", band, len(visits_for_star_band))

        # ── ONE query_datasets call: all PVI refs covering this star in this band
        # No visit filter here — we retrieve all visits at once and match locally.
        try:
            refs = list(
                butler.query_datasets(
                    PVI_DATASET,
                    where=(
                        "band.name = :band "
                        "AND visit.timespan OVERLAPS :timespan "
                        "AND visit_detector_region.region OVERLAPS POINT(:ra, :dec)"
                    ),
                    bind={"band": band, "timespan": timespan, "ra": ra_star, "dec": dec_star},
                    order_by=["visit.timespan.begin"],
                )
            )
        except Exception as exc:
            log.warning("    query_datasets failed (star=%s band=%s): %s", star_id, band, exc)
            n_stars_err += 1
            continue

        log.info("    query returned %d refs (expect ~%d)", len(refs), len(visits_for_star_band))

        # Build visit → ref lookup from the query results
        # ref.dataId["visit"] is the authoritative visit number from the registry
        visit_to_ref = {int(ref.dataId["visit"]): ref for ref in refs}

        # ── Iterate over the rows of this star × band ─────────────────────────
        for idx in df_all.index[mask_band]:
            visit = int(df_all.at[idx, "visit"])
            x = float(df_all.at[idx, "x"])
            y = float(df_all.at[idx, "y"])

            ref = visit_to_ref.get(visit)
            if ref is None:
                log.debug("    visit %d not found in query results — skipping", visit)
                continue

            # Detector id from the registry dataId (reliable)
            det_id_from_ref = int(ref.dataId["detector"])
            cache_key = (band, visit, det_id_from_ref)

            # Load PVI only if not already cached
            if cache_key not in calib_cache:
                log.debug("    Loading PVI  band=%s  visit=%d  detector=%d", band, visit, det_id_from_ref)
                calib_cache[cache_key] = load_pvi_calib(butler, ref)
                gc.collect()

            cdict = calib_cache[cache_key]
            if not cdict["calib_ok"]:
                continue

            # Write CCD-level scalars
            df_all.at[idx, "calib_mean"] = cdict["calib_mean"]
            df_all.at[idx, "calib_err"] = cdict["calib_err"]
            df_all.at[idx, "zeropoint"] = cdict["zeropoint"]
            df_all.at[idx, "visit_from_image"] = cdict["visit_from_image"]
            df_all.at[idx, "detector_id"] = cdict["detector_id"]
            df_all.at[idx, "detector_name"] = cdict["detector_name"]

            # Local calibration at the source pixel position
            try:
                pos = Point2D(x, y)
                df_all.at[idx, "calib_local"] = cdict["photocal"].getLocalCalibration(pos)
            except Exception as exc:
                log.debug("    getLocalCalibration failed at idx=%d: %s", idx, exc)

    n_stars_ok += 1

log.info("Star loop done: %d OK, %d with query errors.", n_stars_ok, n_stars_err)
log.info("Total unique PVIs loaded (cache size): %d", len(calib_cache))

## 8. Sanity check: visit number consistency

In [ ]:
mask_ok = df_all["visit_from_image"] > 0
mismatches = df_all.loc[
    mask_ok & (df_all["visit"] != df_all["visit_from_image"]),
    ["simbad_id", "band", "visit", "visit_from_image", "detector_id"],
]

if len(mismatches) > 0:
    print(f"WARNING — {len(mismatches)} visit mismatches found:")
    display(mismatches)
else:
    log.info("All visit numbers consistent between query and visitInfo.id — OK.")

## 9. Save the global enriched file

In [ ]:
def save_calib(df: pd.DataFrame, out_dir: str, basename: str) -> None:
    """Save enriched DataFrame as both CSV and Parquet."""
    csv_path = os.path.join(out_dir, basename + ".csv")
    parquet_path = os.path.join(out_dir, basename + ".parquet")
    df.to_csv(csv_path, index=False)
    df.to_parquet(parquet_path, index=False)
    log.info("Saved CSV    : %s  (%d rows)", csv_path, len(df))
    log.info("Saved Parquet: %s", parquet_path)

In [ ]:
save_calib(df_all, DIR_DATA, "all_stars_lightcurves_calib")

# Quick preview of the new columns
df_all[
    [
        "simbad_id",
        "visit",
        "band",
        "detector_id",
        "detector_name",
        "calib_mean",
        "calib_err",
        "calib_local",
        "zeropoint",
        "visit_from_image",
    ]
].head(8)

## 10. Generate and save per-star files from the enriched global table

In [ ]:
# Re-read the per-star input filenames to preserve the original naming convention
per_star_files = sorted(f for f in os.listdir(DIR_LC_PER_STAR_IN) if f.endswith("_lc_mjd.csv"))
log.info("Found %d per-star CSV files in %s", len(per_star_files), DIR_LC_PER_STAR_IN)

n_ok_ps = 0
n_err_ps = 0

for fname in per_star_files:
    src_path = os.path.join(DIR_LC_PER_STAR_IN, fname)
    try:
        df_star_orig = pd.read_csv(src_path)
        df_star_orig["visit"] = df_star_orig["visit"].astype(np.int64)
    except Exception as exc:
        log.error("ERROR reading %s: %s", fname, exc)
        n_err_ps += 1
        continue

    # Identify the star(s) in this file
    star_ids_in_file = df_star_orig["simbad_id"].unique()

    # Pull the enriched rows from the global table (already computed above)
    df_star_calib = df_all.loc[
        df_all["simbad_id"].isin(star_ids_in_file) & df_all["visit"].isin(df_star_orig["visit"])
    ].copy()

    stem = fname.replace("_lc_mjd.csv", "")
    out_stem = stem + "_lc_calib"
    save_calib(df_star_calib, DIR_DATA_PER_STAR, out_stem)
    n_ok_ps += 1

log.info("Per-star files saved: %d OK, %d errors.", n_ok_ps, n_err_ps)

## 11. Diagnostics

In [ ]:
# Calibration statistics per band
calib_summary = (
    df_all.dropna(subset=["calib_mean"])
    .groupby("band")
    .agg(
        n_rows=("calib_mean", "count"),
        calib_mean_med=("calib_mean", "median"),
        calib_mean_std=("calib_mean", "std"),
        calib_local_med=("calib_local", "median"),
        zeropoint_med=("zeropoint", "median"),
    )
)
print("Calibration summary per band:")
display(calib_summary)

In [ ]:
# Rows where calib retrieval failed
n_missing = df_all["calib_mean"].isna().sum()
n_total = len(df_all)
log.info(
    "Rows without calib (Butler call failed or visit not in query): " "%d / %d  (%.1f %%)",
    n_missing,
    n_total,
    100.0 * n_missing / n_total if n_total else 0,
)

# Detectors actually assigned per band
print("\nDetector ids found per band:")
display(
    df_all.dropna(subset=["calib_mean"])
    .groupby("band")["detector_id"]
    .value_counts()
    .rename("n_rows")
    .reset_index()
    .sort_values(["band", "detector_id"])
)

## 12. Quick-look plots

In [ ]:
def savefig(fig, name, dpi=150):
    """Save figure as both PDF and PNG."""
    fig.savefig(os.path.join(DIR_FIGS, name + ".pdf"), dpi=dpi, bbox_inches="tight")
    fig.savefig(os.path.join(DIR_FIGS, name + ".png"), dpi=dpi, bbox_inches="tight")
    log.info("Figure saved: %s (.pdf/.png)", os.path.join(DIR_FIGS, name))

In [ ]:
# ── Plot 1: distribution of calib_mean per band ───────────────────────────────
bands_available = sorted(df_all["band"].dropna().unique())
n_bands = len(bands_available)
cmap = plt.get_cmap("tab10")

if n_bands > 0:
    fig, axes = plt.subplots(1, n_bands, figsize=(4 * n_bands, 4), sharey=False)
    if n_bands == 1:
        axes = [axes]
    for ax, band in zip(axes, bands_available):
        vals = df_all.loc[df_all["band"] == band, "calib_mean"].dropna()
        color = cmap(bands_available.index(band))
        ax.hist(vals, bins=40, color=color, alpha=0.75, edgecolor="white")
        ax.set_title(f"band = {band}")
        ax.set_xlabel("calib_mean  [nJy / ADU]")
        ax.set_ylabel("count")
    fig.suptitle("Distribution of PhotoCalib mean per band", fontsize=12)
    plt.tight_layout()
    savefig(fig, "calib_mean_distribution_per_band")
    plt.show()

In [ ]:
# ── Plot 2: zero-point vs expMidptMJD per band ────────────────────────────────
if "expMidptMJD" in df_all.columns and n_bands > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    for band in bands_available:
        sub = df_all[(df_all["band"] == band) & df_all["zeropoint"].notna()].drop_duplicates(
            subset=["visit", "detector_id"]
        )
        color = cmap(bands_available.index(band))
        ax.scatter(sub["expMidptMJD"], sub["zeropoint"], s=10, alpha=0.6, label=band, color=color)
    ax.set_xlabel("expMidptMJD")
    ax.set_ylabel("Zero-point  [AB mag]")
    ax.set_title("Photometric zero-point vs. time — one point per (visit, detector)")
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    savefig(fig, "zeropoint_vs_mjd")
    plt.show()

In [ ]:
# ── Plot 3: calib_mean vs expMidptMJD per band ────────────────────────────────
if "expMidptMJD" in df_all.columns and n_bands > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    for band in bands_available:
        sub = df_all[(df_all["band"] == band) & df_all["calib_mean"].notna()].drop_duplicates(
            subset=["visit", "detector_id"]
        )
        color = cmap(bands_available.index(band))
        ax.scatter(sub["expMidptMJD"], sub["calib_mean"], s=10, alpha=0.6, label=band, color=color)
    ax.set_xlabel("expMidptMJD")
    ax.set_ylabel("calib_mean  [nJy / ADU]")
    ax.set_title("PhotoCalib mean vs. time — one point per (visit, detector)")
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    savefig(fig, "calib_mean_vs_mjd")
    plt.show()

In [ ]:
# ── Plot 4: calib_local vs calib_mean (spatial variation across the CCD) ──────
if n_bands > 0:
    fig, ax = plt.subplots(figsize=(6, 6))
    for band in bands_available:
        sub = df_all[(df_all["band"] == band) & df_all["calib_mean"].notna() & df_all["calib_local"].notna()]
        color = cmap(bands_available.index(band))
        ax.scatter(sub["calib_mean"], sub["calib_local"], s=5, alpha=0.4, label=band, color=color)
    lims_min = df_all[["calib_mean", "calib_local"]].min().min()
    lims_max = df_all[["calib_mean", "calib_local"]].max().max()
    ax.plot([lims_min, lims_max], [lims_min, lims_max], "k--", lw=1, label="1:1")
    ax.set_xlabel("calib_mean  [nJy / ADU]")
    ax.set_ylabel("calib_local  [nJy / ADU]")
    ax.set_title("Local vs. mean PhotoCalib factor")
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    savefig(fig, "calib_local_vs_mean")
    plt.show()

## 13. Output directory listing

In [ ]:
print(f"\n=== Contents of {DIR_DATA} ===")
for entry in sorted(os.listdir(DIR_DATA)):
    full = os.path.join(DIR_DATA, entry)
    if os.path.isdir(full):
        n = len(os.listdir(full))
        print(f"  [DIR]  {entry}/  ({n} files)")
    else:
        size_kb = os.path.getsize(full) / 1024
        print(f"  [FILE] {entry}  ({size_kb:.1f} kB)")